In [1]:
!git clone -b second_branch https://github.com/ayeshathoi/CS682-Project.git
%cd CS682-Project

Cloning into 'CS682-Project'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 86 (delta 32), reused 30 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 2.06 MiB | 10.68 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/CS682-Project


In [2]:
!pip install -r requirements.txt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm "/content/CS682-Project/checkpoints_inter/clean_seg_e12_gfi0.3_w64_c8_s0.pt"
!rm "/content/CS682-Project/checkpoints_inter/stego_inter_e12_a20.0_b1.0_gfi0.3_w64_c8_s0.pt"

In [ ]:
!rm -rf /content/CS682-Project/core/__pycache__
!rm -rf /content/CS682-Project/__pycache__

In [6]:
!rm -f /content/CS682-Project/checkpoints_inter/stego_inter_*.pt
!rm -f /content/CS682-Project/checkpoints_inter/clean_seg_*.pt

# Verify the patch is still on disk after restart
!grep -c "_resize_bn_at" /content/CS682-Project/core/gfi_seq.py
# Should print 3

3


##this code works !!

In [7]:
#!/usr/bin/env python3
"""
Inter-task DNN steganography (Li et al., arXiv:2307.03444, Sec. 5.1):
  Secret: DnCNN-style denoiser trained on Oxford-Pet RGB images.
  Stego D_st: Oxford-IIIT Pet trimap segmentation (3 classes).
  Pipeline: GFI -> SIH -> POS with L_st + alpha*L_mu + beta*L_sig.

What changed vs your previous version:
  * Denoising loss is MSE on the residual (predict noise, not clean image).
    Standard DnCNN training recipe (Zhang et al. 2017); converges several
    dB faster than direct L1 on the clean image.
  * Noise is added in **raw [0,1] pixel space** with sigma = noise_level/255,
    matching the paper's "noise level 50" convention.
  * PSNR is computed in raw [0,1] space with MAX=1.0, so numbers are
    directly comparable to literature (DnCNN-S reports ~25.6 dB at sigma=50).
  * Two data pipelines: raw [0,1] for denoising, ImageNet-normalized for seg.
  * Removed duplicated PSNR call; cleaned up indentation.
  * Added per-epoch loss prints + sanity-check PSNR floor.
  * Default --epochs-secret bumped from 8 to 30 (paper assumes pretrained
    DnCNN; we train from scratch on small data so we need more epochs).
  * Checkpointing: secret/clean/stego all skip if already saved.
  * LSB re-embed after stego training (intra-task fix carried over).
"""

from __future__ import annotations

import argparse
import copy
import math
import random
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

from PIL import Image

from core.extract import extract_secret_encoder, relative_param_error_prefix
from core.gfi_seq import (
    apply_insertions_seq,
    compute_position_importance_seq,
    count_total_insertion_positions_seq,
    conv_indices_in_seq,
    top_n_positions,
)
from core.masks import (
    apply_grad_mask_,
    build_partial_masks_dual_encoder,
    channel_roles_after_sih,
    statistical_loss_encoder,
)
from core.sih import insert_side_filter_and_embed_encoder
from data import mean_iou, oxford_pet_seg_loaders
from models.dncnn_dual import DualHeadDnCNN


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def patch_first_conv_in_channels(module, new_in_channels):
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d):
            old = child
            new_conv = nn.Conv2d(
                in_channels=new_in_channels, out_channels=old.out_channels,
                kernel_size=old.kernel_size, stride=old.stride,
                padding=old.padding, dilation=old.dilation,
                groups=old.groups, bias=(old.bias is not None),
            ).to(old.weight.device)
            with torch.no_grad():
                new_conv.weight.zero_()
                c = min(old.in_channels, new_in_channels)
                new_conv.weight[:, :c, :, :] = old.weight[:, :c, :, :]
                if old.bias is not None:
                    new_conv.bias.copy_(old.bias)
            setattr(module, name, new_conv)
            return True
        if patch_first_conv_in_channels(child, new_in_channels):
            return True
    return False


def replace_bn_with_lazy_bn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, nn.LazyBatchNorm2d().to(next(module.parameters()).device))
        else:
            replace_bn_with_lazy_bn(child)


def imagenet_normalize(x: torch.Tensor) -> torch.Tensor:
    """[0,1] -> ImageNet-normalized."""
    return (x - _IMAGENET_MEAN.to(x.device)) / _IMAGENET_STD.to(x.device)


# ---------------------------------------------------------------------------
# Raw [0,1] Oxford-Pet loader for the denoising pathway
# ---------------------------------------------------------------------------

def oxford_pet_raw_loaders(
    data_dir: str, batch_size: int, img_size: int = 128,
    train_n: int = 6000, test_n: int = 100, seed: int = 0,
    num_workers: int = 2,
) -> tuple[DataLoader, DataLoader]:
    """Loaders that return raw [0,1] RGB tensors. Train adds horizontal flip."""
    base = torchvision.datasets.OxfordIIITPet(
        data_dir, split="trainval", target_types=("segmentation",),
        download=True, transform=None, target_transform=None,
    )
    n_total = len(base)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n_total).tolist()
    if train_n + test_n > n_total:
        train_n = int(n_total * 0.95)
        test_n = n_total - train_n
    train_idx = perm[:train_n]
    # take test from after the typical val region
    test_start = min(train_n + 1282, n_total - test_n)
    test_idx = perm[test_start: test_start + test_n]

    class RawSubset(torch.utils.data.Dataset):
        def __init__(self, indices, augment: bool):
            self.indices = list(indices)
            self.augment = augment

        def __len__(self): return len(self.indices)

        def __getitem__(self, i):
            img, _ = base[self.indices[i]]
            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)
            img = img.convert("RGB")
            img = TF.resize(img, [img_size, img_size], antialias=True)
            if self.augment and random.random() < 0.5:
                img = TF.hflip(img)
            return TF.to_tensor(img)  # in [0,1]

    train_ds = RawSubset(train_idx, augment=True)
    test_ds  = RawSubset(test_idx,  augment=False)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                   num_workers=num_workers, pin_memory=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                   num_workers=num_workers, pin_memory=True),
    )


# ---------------------------------------------------------------------------
# Denoising training / eval — all in [0,1] space
# ---------------------------------------------------------------------------

def train_denoiser(
    model: DualHeadDnCNN, raw_loader: DataLoader, device: torch.device,
    epochs: int, lr: float, noise_level: float, desc: str,
) -> list[float]:
    """MSE on residual: model is asked to output the noise. Clean = noisy - residual."""
    model.train()
    opt = Adam(
        [{"params": model.encoder.parameters()},
         {"params": model.secret_head.parameters()}],
        lr=lr,
    )
    sigma = noise_level / 255.0
    epoch_losses: list[float] = []
    for ep in range(epochs):
        total = 0.0; nb = 0
        pbar = tqdm(raw_loader, desc=f"{desc} ep{ep+1}/{epochs}", leave=False)
        for x in pbar:
            x = x.to(device, non_blocking=True)              # [0,1]
            noise = torch.randn_like(x) * sigma
            noisy01 = x + noise
            # Encoder was implicitly conditioned on ImageNet stats via seg path,
            # so feed normalized inputs here too for distribution consistency.
            inp = imagenet_normalize(noisy01)
            pred_residual = model.forward_secret(inp)        # 3xHxW
            loss = F.mse_loss(pred_residual, noise)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.item(); nb += 1
            pbar.set_postfix(loss=loss.item())
        avg = total / max(nb, 1)
        epoch_losses.append(avg)
        print(f"{desc} ep {ep+1}/{epochs}: avg MSE={avg:.5f}")
    return epoch_losses


@torch.no_grad()
def average_psnr_denoise(
    model: DualHeadDnCNN, raw_loader: DataLoader, device: torch.device,
    noise_level: float,
) -> float:
    """Per-image PSNR with MAX=1.0 on [0,1]-clipped denoised output."""
    model.eval()
    sigma = noise_level / 255.0
    total_psnr = 0.0; n_imgs = 0
    for x in raw_loader:
        x = x.to(device, non_blocking=True)
        noise = torch.randn_like(x) * sigma
        noisy01 = x + noise
        inp = imagenet_normalize(noisy01)
        pred_residual = model.forward_secret(inp)
        denoised01 = (noisy01 - pred_residual).clamp(0.0, 1.0)
        mse = (denoised01 - x).pow(2).mean(dim=(1, 2, 3)).clamp(min=1e-10)
        psnr = 10.0 * torch.log10(1.0 / mse)
        total_psnr += psnr.sum().item()
        n_imgs += x.shape[0]
    return total_psnr / max(n_imgs, 1)


@torch.no_grad()
def baseline_psnr_noisy(raw_loader: DataLoader, device: torch.device,
                        noise_level: float) -> float:
    """PSNR floor: noisy input vs clean target, no model."""
    sigma = noise_level / 255.0
    total = 0.0; n = 0
    for x in raw_loader:
        x = x.to(device, non_blocking=True)
        noise = torch.randn_like(x) * sigma
        noisy = (x + noise).clamp(0.0, 1.0)
        mse = (noisy - x).pow(2).mean(dim=(1, 2, 3)).clamp(min=1e-10)
        psnr = 10.0 * torch.log10(1.0 / mse)
        total += psnr.sum().item(); n += x.shape[0]
    return total / max(n, 1)


# ---------------------------------------------------------------------------
# Segmentation training
# ---------------------------------------------------------------------------

def train_seg_full(model: DualHeadDnCNN, loader: DataLoader,
                   device: torch.device, epochs: int, lr: float, desc: str) -> list[float]:
    model.train()
    opt = Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    losses: list[float] = []
    for ep in range(epochs):
        total = 0.0; nb = 0
        pbar = tqdm(loader, desc=f"{desc} ep{ep+1}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = model.forward_stego(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
            total += loss.item(); nb += 1
            pbar.set_postfix(loss=loss.item())
        avg = total / max(nb, 1)
        losses.append(avg)
        print(f"{desc} ep {ep+1}/{epochs}: avg CE={avg:.4f}")
    return losses


def train_stego_partial(
    stego: DualHeadDnCNN, grad_masks: dict, clean_ref: DualHeadDnCNN,
    loader: DataLoader, device: torch.device, epochs: int, lr: float,
    alpha: float, beta: float, desc: str,
) -> list[float]:
    stego.train(); clean_ref.eval()
    opt = Adam(stego.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    losses: list[float] = []
    for ep in range(epochs):
        total = 0.0; nb = 0
        pbar = tqdm(loader, desc=f"{desc} ep{ep+1}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = stego.forward_stego(x)
            l_st = crit(logits, y)
            l_mu, l_sig = statistical_loss_encoder(stego, clean_ref, "encoder")
            loss = l_st + alpha * l_mu + beta * l_sig
            loss.backward()
            apply_grad_mask_(stego, grad_masks)
            opt.step()
            total += loss.item(); nb += 1
            pbar.set_postfix(loss=l_st.item())
        avg = total / max(nb, 1)
        losses.append(avg)
        print(f"{desc} ep {ep+1}/{epochs}: avg total={avg:.4f}")
    return losses


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=str, default=str(ROOT / "data"))
    ap.add_argument("--ckpt-dir", type=str, default=str(ROOT / "checkpoints_inter"))
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", type=str,
                    default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--img-size", type=int, default=128)
    ap.add_argument("--epochs-secret", type=int, default=30,
                    help="Bumped from 8: paper assumes pretrained DnCNN.")
    ap.add_argument("--epochs-clean", type=int, default=12)
    ap.add_argument("--epochs-stego", type=int, default=12)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--alpha", type=float, default=20.0)
    ap.add_argument("--beta", type=float, default=1.0)
    ap.add_argument("--noise-level", type=float, default=50.0,
                    help="Gaussian noise level on raw [0,255] (paper: 50).")
    ap.add_argument("--gfi-fraction", type=float, default=0.30)
    ap.add_argument("--gfi-max-batches", type=int, default=40)
    ap.add_argument("--encoder-convs", type=int, default=8)
    ap.add_argument("--encoder-width", type=int, default=64)
    ap.add_argument("--key", type=str, default="cs682-inter-task-secret-key!")
    ap.add_argument("--force-retrain", action="store_true")
    ap.add_argument("--force-retrain-stego", action="store_true")
    args, unknown = ap.parse_known_args()
    if unknown:
        print(f"Warning: ignoring unexpected arguments: {unknown}")

    set_seed(args.seed)
    device = torch.device(args.device)
    ckpt_dir = Path(args.ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    print(f"[ckpt] {ckpt_dir}")

    # ---- Two data pipelines ----
    pet_train_seg, _pet_val, pet_test_seg, num_seg_classes = oxford_pet_seg_loaders(
        args.data_dir, args.batch_size, img_size=args.img_size, seed=args.seed,
    )
    pet_train_raw, pet_test_raw = oxford_pet_raw_loaders(
        args.data_dir, args.batch_size, img_size=args.img_size, seed=args.seed,
    )

    # ============== Phase 1: Secret denoiser ==============
    secret_ckpt = ckpt_dir / (
        f"secret_dncnn_e{args.epochs_secret}_n{int(args.noise_level)}"
        f"_w{args.encoder_width}_c{args.encoder_convs}_s{args.seed}.pt"
    )

    secret = DualHeadDnCNN(
        num_seg_classes=num_seg_classes,
        num_encoder_convs=args.encoder_convs,
        width=args.encoder_width,
        init_weights=True,
    ).to(device)

    if secret_ckpt.exists() and not args.force_retrain:
        print(f"[secret] loading {secret_ckpt.name}")
        blob = torch.load(secret_ckpt, map_location=device, weights_only=False)
        secret.load_state_dict(blob["model"])
    else:
        floor = baseline_psnr_noisy(pet_test_raw, device, args.noise_level)
        print(f"[sanity] PSNR floor (no denoiser): {floor:.2f} dB")
        print("Training secret denoiser (residual MSE in [0,1] space)...")
        losses = train_denoiser(
            secret, pet_train_raw, device,
            args.epochs_secret, args.lr, args.noise_level, "secret denoise",
        )
        torch.save({"model": secret.state_dict(), "losses": losses,
                    "noise_level": args.noise_level,
                    "epochs": args.epochs_secret}, secret_ckpt)
        print(f"[secret] saved {secret_ckpt.name}")

    ps_before = average_psnr_denoise(secret, pet_test_raw, device, args.noise_level)
    print(f"Secret denoiser PSNR (sigma={args.noise_level}/255, MAX=1): "
          f"{ps_before:.2f} dB")

    # ============== Phase 2: GFI on segmentation probe ==============
    # probe = copy.deepcopy(secret).to(device)
    # crit_seg = nn.CrossEntropyLoss()

    # def loss_forward_seg(m, xb, yb):
    #     return crit_seg(m.forward_stego(xb), yb)

    # max_b = None if args.gfi_max_batches == 0 else args.gfi_max_batches
    # print("Computing GFI position importance (seg loss, normalized inputs)...")
    # imp = compute_position_importance_seq(
    #     probe, "encoder", pet_train_seg, device, loss_forward_seg, max_batches=max_b,
    # )
    # total_pos = count_total_insertion_positions_seq(secret.encoder)
    # n_sel = max(1, int(round(args.gfi_fraction * total_pos)))
    # specs_all = top_n_positions(imp, n_sel)
    # n_cr = len(conv_indices_in_seq(secret.encoder))
    # specs = [s for s in specs_all if s.conv_rank < n_cr - 1]
    # print(f"slots={total_pos}, top-N={n_sel}, kept={len(specs)} (excluded last conv)")


    # ============== Phase 4 setup: build stego architecture ==============
    stego_ckpt = ckpt_dir / (
        f"stego_inter_e{args.epochs_stego}_a{args.alpha}_b{args.beta}"
        f"_gfi{args.gfi_fraction}_w{args.encoder_width}"
        f"_c{args.encoder_convs}_s{args.seed}.pt"
    )

    key = args.key.encode("utf-8")[:64]
    if len(key) < 8:
        key = key + b"\0" * (8 - len(key))

    stego_exists = stego_ckpt.exists() and not (
        args.force_retrain or args.force_retrain_stego
    )

    if stego_exists:
        # Replay the stored specs so we rebuild the same architecture.
        print(f"[stego] preparing to load {stego_ckpt.name}")
        blob = torch.load(stego_ckpt, map_location="cpu", weights_only=False)
        if "specs" not in blob:
            raise RuntimeError(
                f"Checkpoint {stego_ckpt.name} predates the specs-saving fix. "
                "Delete it and retrain (it'll be saved correctly this time).")
        specs = blob["specs"]
        conv_masks = blob["conv_masks"]
        side_info = blob["side_info"]
        gfi_shapes = blob["gfi_shapes"]
        print(f"  replayed {len(specs)} insertion positions from checkpoint")
    else:
        # Fresh run: compute GFI now.
        probe = copy.deepcopy(secret).to(device)
        crit_seg = nn.CrossEntropyLoss()

        def loss_forward_seg(m, xb, yb):
            return crit_seg(m.forward_stego(xb), yb)

        max_b = None if args.gfi_max_batches == 0 else args.gfi_max_batches
        print("Computing GFI position importance (seg loss, normalized inputs)...")
        imp = compute_position_importance_seq(
            probe, "encoder", pet_train_seg, device,
            loss_forward_seg, max_batches=max_b,
        )
        total_pos = count_total_insertion_positions_seq(secret.encoder)
        n_sel = max(1, int(round(args.gfi_fraction * total_pos)))
        specs_all = top_n_positions(imp, n_sel)
        n_cr = len(conv_indices_in_seq(secret.encoder))
        specs = [s for s in specs_all if s.conv_rank < n_cr - 1]
        print(f"slots={total_pos}, top-N={n_sel}, kept={len(specs)} "
              f"(excluded last conv)")
        conv_masks = None  # filled in below

    # ---- Build the stego architecture from `specs` ----
    rng = torch.Generator(device="cpu").manual_seed(args.seed)
    stego_base = copy.deepcopy(secret)
    new_enc, conv_masks_built = apply_insertions_seq(stego_base.encoder, specs, rng)
    stego_base.encoder = new_enc
    if conv_masks is None:
        conv_masks = conv_masks_built

    stego_model, side_info_built = insert_side_filter_and_embed_encoder(
        stego_base, "encoder", conv_masks, key, rng,
    )
    stego_model = stego_model.to(device)
    replace_bn_with_lazy_bn(stego_model)

    # Patch stego_head input channels after SIH widened the encoder
    stego_model.eval()
    with torch.no_grad():
        x_dummy_seg, _ = next(iter(pet_train_seg))
        x_dummy_seg = x_dummy_seg[:1].to(device)
        enc_out = stego_model.encoder(x_dummy_seg)
        new_channels = enc_out.shape[1]

    if isinstance(stego_model.stego_head, nn.Conv2d):
        old = stego_model.stego_head
        new_head = nn.Conv2d(
            in_channels=new_channels, out_channels=old.out_channels,
            kernel_size=old.kernel_size, stride=old.stride,
            padding=old.padding, dilation=old.dilation,
            groups=old.groups, bias=(old.bias is not None),
        ).to(device)
        with torch.no_grad():
            new_head.weight.zero_()
            c = min(old.in_channels, new_channels)
            new_head.weight[:, :c, :, :] = old.weight[:, :c, :, :]
            if old.bias is not None:
                new_head.bias.copy_(old.bias)
        stego_model.stego_head = new_head
    else:
        patch_first_conv_in_channels(stego_model.stego_head, new_channels)

    stego_model.eval()
    with torch.no_grad():
        _ = stego_model.forward_stego(x_dummy_seg)

    if not stego_exists:
        gfi_shapes = {r: int(conv_masks[r].numel())
                      for r in sorted(conv_masks.keys())}
        side_info = side_info_built

    roles = channel_roles_after_sih(conv_masks, side_info)
    grad_masks = build_partial_masks_dual_encoder(stego_model, roles, "encoder")

    # ============== Phase 3: Clean reference (seg) ==============
    clean_ckpt = ckpt_dir / (
        f"clean_seg_e{args.epochs_clean}_gfi{args.gfi_fraction}"
        f"_w{args.encoder_width}_c{args.encoder_convs}_s{args.seed}.pt"
    )
    clean = copy.deepcopy(stego_model)
    clean._initialize_weights()
    clean = clean.to(device)

    if clean_ckpt.exists() and not args.force_retrain:
        print(f"[clean] loading {clean_ckpt.name}")
        blob = torch.load(clean_ckpt, map_location=device, weights_only=False)
        clean.load_state_dict(blob["model"])
    else:
        print("Training clean reference (full weights, segmentation)...")
        losses = train_seg_full(clean, pet_train_seg, device,
                                args.epochs_clean, args.lr, "clean seg")
        torch.save({"model": clean.state_dict(), "losses": losses,
                    "epochs": args.epochs_clean}, clean_ckpt)

    mi_clean = mean_iou(clean, pet_test_seg, device, num_seg_classes)
    print(f"Clean ref test mIoU: {100.0 * mi_clean:.2f}%")


    # ============== Phase 4: Stego (partial updates) ==============
    if stego_exists:
        print(f"[stego] loading {stego_ckpt.name}")
        stego_model.load_state_dict(blob["model"])
    else:
        print("Training stego G_delta (partial updates + statistical losses)...")
        stego_losses = train_stego_partial(
            stego_model, grad_masks, clean, pet_train_seg, device,
            args.epochs_stego, args.lr, args.alpha, args.beta, "stego seg",
        )

        # ---- Re-embed LSB payload (intra-task fix carried over) ----
        from core.sih import pack_conv_masks, encrypt_payload, embed_bits_lsb_
        enc = stego_model.encoder
        fi = conv_indices_in_seq(enc)[side_info.conv_rank]
        side_conv = enc[fi]
        payload = pack_conv_masks(conv_masks)
        enc_payload = encrypt_payload(payload, key)
        bits = []
        for b in enc_payload:
            for j in range(8):
                bits.append((b >> j) & 1)
        used = embed_bits_lsb_(side_conv.weight.data, bits, 0)
        if side_conv.bias is not None and used < len(bits):
            embed_bits_lsb_(side_conv.bias.data, bits, used)
        print(f"  [sih] re-embedded {len(bits)} LSB bits in side conv")

        torch.save({
            "model": stego_model.state_dict(),
            "losses": stego_losses,
            "specs": specs,
            "side_info": side_info,
            "gfi_shapes": gfi_shapes,
            "conv_masks": {r: conv_masks[r] for r in sorted(conv_masks.keys())},
            "epochs": args.epochs_stego, "lr": args.lr,
            "alpha": args.alpha, "beta": args.beta,
        }, stego_ckpt)
        print(f"[stego] saved {stego_ckpt.name}")

    mi_stego = mean_iou(stego_model, pet_test_seg, device, num_seg_classes)
    print(f"Stego test mIoU: {100.0 * mi_stego:.2f}%")

#     # ============== Phase 4: Stego (partial updates) ==============
#     if stego_exists:
#         print(f"[stego] loading {stego_ckpt.name}")
#         stego_model.load_state_dict(blob["model"])
#     else:
#         print("Training stego G_delta (partial updates + statistical losses)...")
#         stego_losses = train_stego_partial(
#             stego_model, grad_masks, clean, pet_train_seg, device,
#             args.epochs_stego, args.lr, args.alpha, args.beta, "stego seg",
#         )
#         #print("Training stego G_delta (partial updates + statistical losses)...")

# #         torch.save({
# #     "model": stego_model.state_dict(),
# #     "specs": specs,
# #     "side_info": side_info,
# #     "gfi_shapes": gfi_shapes,
# #     "conv_masks": {r: conv_masks[r] for r in sorted(conv_masks.keys())},
# #     "epochs": args.epochs_stego, "lr": args.lr,
# #     "alpha": args.alpha, "beta": args.beta,
# # }, stego_ckpt)


#         # ---- Re-embed LSB payload (intra-task fix carried over) ----
#         from core.sih import pack_conv_masks, encrypt_payload, embed_bits_lsb_
#         enc = stego_model.encoder
#         fi = conv_indices_in_seq(enc)[side_info.conv_rank]
#         side_conv = enc[fi]
#         payload = pack_conv_masks(conv_masks)
#         enc_payload = encrypt_payload(payload, key)
#         bits = []
#         for b in enc_payload:
#             for j in range(8):
#                 bits.append((b >> j) & 1)
#         used = embed_bits_lsb_(side_conv.weight.data, bits, 0)
#         if side_conv.bias is not None and used < len(bits):
#             embed_bits_lsb_(side_conv.bias.data, bits, used)
#         print(f"  [sih] re-embedded {len(bits)} LSB bits in side conv")

#         torch.save({
#             "model": stego_model.state_dict(),
#             "specs": specs,
#             "side_info": side_info,
#             "gfi_shapes": gfi_shapes,
#             "conv_masks": {r: conv_masks[r] for r in sorted(conv_masks.keys())},
#             "epochs": args.epochs_stego, "lr": args.lr,
#             "alpha": args.alpha, "beta": args.beta,
#         }, stego_ckpt)
#         print(f"[stego] saved {stego_ckpt.name}")

#     mi_stego = mean_iou(stego_model, pet_test_seg, device, num_seg_classes)
#     print(f"Stego test mIoU: {100.0 * mi_stego:.2f}%")


    # ============== Phase 5: Recovery ==============
    recovered = extract_secret_encoder(stego_model.cpu(), side_info, key, gfi_shapes).to(device)
    recovered.eval()  # BN stats inherited correctly from the trained secret

    ps_rec = average_psnr_denoise(recovered, pet_test_raw, device, args.noise_level)
    print(f"Recovered secret PSNR: {ps_rec:.2f} dB")
    print(f"Recovery delta: {ps_rec - ps_before:+.2f} dB")

    drift = relative_param_error_prefix(
        secret.cpu(), recovered.cpu(), ("encoder.", "secret_head.")
    )
    print(f"Relative L1 drift on encoder+secret_head: {drift:.3e}")

    print("\n=== Summary ===")
    print(f"  Secret PSNR:    {ps_before:.2f} dB")
    print(f"  Recovered PSNR: {ps_rec:.2f} dB")
    print(f"  Stego mIoU:     {100*mi_stego:.2f} %")
    print(f"  Clean mIoU:     {100*mi_clean:.2f} %")
    print(f"  Drift:          {drift:.3e}")
    print("Done.")

    # ============== Phase 5: Recovery ==============
    # recovered = extract_secret_encoder(stego_model.cpu(), side_info, key, gfi_shapes).to(device)

    # # Determine actual output channels of the recovered encoder
    # # Do a dummy forward pass to get the output shape of the recovered encoder
    # recovered.eval() # ensure layers are in eval mode for shape inference without updating stats
    # with torch.no_grad():
    #     x_dummy_raw = next(iter(pet_train_raw))
    #     x_dummy_raw = x_dummy_raw[:1].to(device) # use a single batch item
    #     dummy_encoder_input = imagenet_normalize(x_dummy_raw)
    #     enc_output_dummy = recovered.encoder(dummy_encoder_input)
    #     actual_encoder_output_channels = enc_output_dummy.shape[1]

    # # The secret_head expects 'width' channels from the encoder
    # expected_secret_head_input_channels = args.encoder_width

    # if actual_encoder_output_channels != expected_secret_head_input_channels:
    #     print(f"  [repair] patching recovered secret_head: encoder output has {actual_encoder_output_channels} channels, but secret_head expects {expected_secret_head_input_channels}.")
    #     old_secret_head = recovered.secret_head
    #     new_secret_head = nn.Conv2d(
    #         in_channels=actual_encoder_output_channels,
    #         out_channels=old_secret_head.out_channels,
    #         kernel_size=old_secret_head.kernel_size,
    #         stride=old_secret_head.stride,
    #         padding=old_secret_head.padding,
    #         dilation=old_secret_head.dilation,
    #         groups=old_secret_head.groups,
    #         bias=(old_secret_head.bias is not None),
    #     ).to(device)
    #     with torch.no_grad():
    #         # Copy as much of the old weight as possible. Weights are [out_channels, in_channels, kH, kW]
    #         c_in_copy = min(old_secret_head.in_channels, actual_encoder_output_channels)
    #         new_secret_head.weight[:, :c_in_copy, :, :] = old_secret_head.weight[:, :c_in_copy, :, :]
    #         # Zero out any new channels if actual_encoder_output_channels > old_secret_head.in_channels
    #         if actual_encoder_output_channels > old_secret_head.in_channels:
    #             new_secret_head.weight[:, old_secret_head.in_channels:, :, :].zero_()

    #         if old_secret_head.bias is not None:
    #             new_secret_head.bias.copy_(old_secret_head.bias)
    #     recovered.secret_head = new_secret_head
    # else:
    #     print(f"  [repair] recovered secret_head input channels match ({actual_encoder_output_channels}). No patching needed.")

    # # ---- Repair BN layers that didn't get shrunk by remove_filter_seq ----
    # def repair_encoder_bn_(model, device):
    #     """Resize any BatchNorm2d in the encoder whose num_features doesn't match
    #     the preceding Conv2d's out_channels. Fresh BN defaults are written; stats
    #     are recalibrated below."""
    #     enc = model.encoder
    #     fixed = 0
    #     last_conv_out = None
    #     for i, m in enumerate(enc):
    #         if isinstance(m, nn.Conv2d):
    #             last_conv_out = m.out_channels
    #         elif isinstance(m, nn.BatchNorm2d):
    #             if last_conv_out is not None and m.num_features != last_conv_out:
    #                 new_bn = nn.BatchNorm2d(last_conv_out).to(device)
    #                 # leave at default init: weight=1, bias=0, running_mean=0,
    #                 # running_var=1, num_batches_tracked=0
    #                 enc[i] = new_bn
    #                 fixed += 1
    #     return fixed

    # n_fixed = repair_encoder_bn_(recovered, device)
    # if n_fixed > 0:
    #     print(f"  [repair] resized {n_fixed} BatchNorm layers in recovered encoder")

    # # ---- Recalibrate BN running stats ----
    # # Put the encoder in train mode so BN updates running stats while we feed
    # # it a few batches of noisy inputs (matching what the secret saw during
    # # training). Keep secret_head in eval mode (it has no BN).
    # sigma = args.noise_level / 255.0
    # recovered.train()
    # with torch.no_grad():
    #     n_calib_batches = 50
    #     for i, x in enumerate(pet_train_raw):
    #         if i >= n_calib_batches:
    #             break
    #         x = x.to(device, non_blocking=True)
    #         noisy01 = x + torch.randn_like(x) * sigma
    #         inp = imagenet_normalize(noisy01)
    #         # Run through encoder only; we don't care about secret_head output here.
    #         _ = recovered.encoder(inp)
    # print(f"  [repair] recalibrated BN stats with {n_calib_batches} batches")

    # recovered.eval()

    # ps_rec = average_psnr_denoise(recovered, pet_test_raw, device, args.noise_level)
    # print(f"Recovered secret PSNR: {ps_rec:.2f} dB")
    # print(f"Recovery delta: {ps_rec - ps_before:+.2f} dB")

    # drift = relative_param_error_prefix(
    #     secret.cpu(), recovered.cpu(), ("encoder.", "secret_head.")
    # )
    # print(f"Relative L1 drift on encoder+secret_head: {drift:.3e}")

    # print("\n=== Summary ===")
    # print(f"  Secret PSNR:    {ps_before:.2f} dB")
    # print(f"  Recovered PSNR: {ps_rec:.2f} dB")
    # print(f"  Stego mIoU:     {100*mi_stego:.2f} %")
    # print(f"  Clean mIoU:     {100*mi_clean:.2f} %")
    # print(f"  Drift:          {drift:.3e}")
    # print("Done.")


if __name__ == "__main__":
    main()

[ckpt] /content/CS682-Project/checkpoints_inter


100%|██████████| 792M/792M [00:03<00:00, 219MB/s]
100%|██████████| 19.2M/19.2M [00:00<00:00, 104MB/s] 


[sanity] PSNR floor (no denoiser): 15.01 dB
Training secret denoiser (residual MSE in [0,1] space)...


secret denoise ep 1/30: avg MSE=0.40934


secret denoise ep 2/30: avg MSE=0.03227


...

stego seg ep 8/12: avg total=0.6841


stego seg ep 9/12: avg total=0.6724


stego seg ep 10/12: avg total=0.6663


stego seg ep 11/12: avg total=0.6620


stego seg ep 12/12: avg total=0.6556
  [sih] re-embedded 664 LSB bits in side conv
[stego] saved stego_inter_e12_a20.0_b1.0_gfi0.3_w64_c8_s0.pt
Stego test mIoU: 43.56%
Recovered secret PSNR: 21.70 dB
Recovery delta: -3.63 dB
Relative L1 drift on encoder+secret_head: 3.071e-03

=== Summary ===
  Secret PSNR:    25.33 dB
  Recovered PSNR: 21.70 dB
  Stego mIoU:     43.56 %
  Clean mIoU:     52.77 %
  Drift:          3.071e-03
Done.


Warning: ignoring unexpected arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-fc70eebf-d84a-4e32-ab82-19d0ec9e9e1a.json']
[ckpt] /content/CS682-Project/checkpoints_inter
100%|██████████| 792M/792M [00:03<00:00, 219MB/s]
100%|██████████| 19.2M/19.2M [00:00<00:00, 104MB/s]
[sanity] PSNR floor (no denoiser): 15.01 dB
Training secret denoiser (residual MSE in [0,1] space)...
secret denoise ep 1/30: avg MSE=0.40934
secret denoise ep 2/30: avg MSE=0.03227
secret denoise ep 3/30: avg MSE=0.02047
secret denoise ep 4/30: avg MSE=0.01389
secret denoise ep 5/30: avg MSE=0.01056
secret denoise ep 6/30: avg MSE=0.00868
secret denoise ep 7/30: avg MSE=0.00755
secret denoise ep 8/30: avg MSE=0.00672
secret denoise ep 9/30: avg MSE=0.00614
secret denoise ep 10/30: avg MSE=0.00572
secret denoise ep 11/30: avg MSE=0.00543
secret denoise ep 12/30: avg MSE=0.00512
secret denoise ep 13/30: avg MSE=0.00489
secret denoise ep 14/30: avg MSE=0.00476
secret denoise ep 15/30: avg MSE=0.00456
secret denoise ep 16/30: avg MSE=0.00446
secret denoise ep 17/30: avg MSE=0.00428
secret denoise ep 18/30: avg MSE=0.00416
secret denoise ep 19/30: avg MSE=0.00409
secret denoise ep 20/30: avg MSE=0.00401
secret denoise ep 21/30: avg MSE=0.00389
secret denoise ep 22/30: avg MSE=0.00388
secret denoise ep 23/30: avg MSE=0.00381
secret denoise ep 24/30: avg MSE=0.00369
secret denoise ep 25/30: avg MSE=0.00364
secret denoise ep 26/30: avg MSE=0.00357
secret denoise ep 27/30: avg MSE=0.00356
secret denoise ep 28/30: avg MSE=0.00348
secret denoise ep 29/30: avg MSE=0.00334
secret denoise ep 30/30: avg MSE=0.00328
[secret] saved secret_dncnn_e30_n50_w64_c8_s0.pt
Secret denoiser PSNR (sigma=50.0/255, MAX=1): 25.33 dB
Computing GFI position importance (seg loss, normalized inputs)...
slots=520, top-N=156, kept=151 (excluded last conv)
Training clean reference (full weights, segmentation)...
clean seg ep 1/12: avg CE=0.9151
clean seg ep 2/12: avg CE=0.7240
clean seg ep 3/12: avg CE=0.6861
clean seg ep 4/12: avg CE=0.6593
clean seg ep 5/12: avg CE=0.6401
clean seg ep 6/12: avg CE=0.6099
clean seg ep 7/12: avg CE=0.6020
clean seg ep 8/12: avg CE=0.5863
clean seg ep 9/12: avg CE=0.5686
clean seg ep 10/12: avg CE=0.5704
clean seg ep 11/12: avg CE=0.5481
                                                                              clean seg ep 12/12: avg CE=0.5358
Clean ref test mIoU: 52.77%
Training stego G_delta (partial updates + statistical losses)...
stego seg ep 1/12: avg total=0.9088
stego seg ep 2/12: avg total=0.8127
stego seg ep 3/12: avg total=0.7730
stego seg ep 4/12: avg total=0.7496
stego seg ep 5/12: avg total=0.7296
stego seg ep 6/12: avg total=0.7160
stego seg ep 7/12: avg total=0.7001
stego seg ep 8/12: avg total=0.6841
stego seg ep 9/12: avg total=0.6724
stego seg ep 10/12: avg total=0.6663
stego seg ep 11/12: avg total=0.6620
stego seg ep 12/12: avg total=0.6556
  [sih] re-embedded 664 LSB bits in side conv
[stego] saved stego_inter_e12_a20.0_b1.0_gfi0.3_w64_c8_s0.pt
Stego test mIoU: 43.56%
Recovered secret PSNR: 21.70 dB
Recovery delta: -3.63 dB
Relative L1 drift on encoder+secret_head: 3.071e-03

=== Summary ===
  Secret PSNR:    25.33 dB
  Recovered PSNR: 21.70 dB
  Stego mIoU:     43.56 %
  Clean mIoU:     52.77 %
  Drift:          3.071e-03
Done.

In [10]:
%%writefile /content/CS682-Project/make_plots_inter.py
#!/usr/bin/env python3
"""make_plots_inter.py — figures for the inter-task steganography run."""

from __future__ import annotations
import argparse
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch


def load_losses(ckpt_path: Path) -> list[float]:
    blob = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return blob.get("losses", [])


def fig_loss_curves(secret_losses, clean_losses, stego_losses, out_path: Path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    ax = axes[0]
    eps = np.arange(1, len(secret_losses) + 1)
    ax.plot(eps, secret_losses, marker="o", color="#1f77b4", linewidth=2)
    ax.set_yscale("log")
    ax.set_title("Secret DnCNN denoiser on Oxford-Pet")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Residual MSE (log scale)")
    ax.grid(alpha=0.3, which="both")

    ax = axes[1]
    eps = np.arange(1, len(clean_losses) + 1)
    ax.plot(eps, clean_losses, marker="s", color="#2ca02c", linewidth=2)
    ax.set_title(r"Clean reference $G_\gamma$ on Oxford-Pet seg")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
    ax.grid(alpha=0.3)

    ax = axes[2]
    eps = np.arange(1, len(stego_losses) + 1)
    ax.plot(eps, stego_losses, marker="^", color="#d62728", linewidth=2,
            label="Total loss")
    ax.set_title(r"Stego $G_\delta$ on Oxford-Pet seg (partial updates)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.grid(alpha=0.3); ax.legend()

    fig.suptitle("Inter-task training dynamics", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


def fig_summary(args, out_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    labels = ["Secret\n(original)", "Recovered\n(extracted)"]
    values = [args.psnr_secret, args.psnr_recovered]
    bars = ax.bar(labels, values, color=["#1f77b4", "#17becf"],
                  edgecolor="black", linewidth=0.6)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.4, f"{v:.2f} dB",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylabel("PSNR (dB)"); ax.set_ylim(0, max(values) + 10)
    ax.set_title("Secret task: denoising fidelity")
    ax.grid(axis="y", alpha=0.3)
    delta_psnr = args.psnr_recovered - args.psnr_secret
    ax.text(0.02, 0.97, f"Recovery loss: {delta_psnr:+.2f} dB",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=10, bbox=dict(boxstyle="round,pad=0.4",
                                   facecolor="white", edgecolor="gray", alpha=0.9))

    ax = axes[1]
    labels = ["Clean reference\n($G_\\gamma$)", "Stego model\n($G_\\delta$)"]
    values = [args.miou_clean, args.miou_stego]
    bars = ax.bar(labels, values, color=["#2ca02c", "#d62728"],
                  edgecolor="black", linewidth=0.6)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5, f"{v:.2f}%",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylabel("Test mIoU (%)"); ax.set_ylim(0, max(values) + 15)
    ax.set_title("Cover task: segmentation utility")
    ax.grid(axis="y", alpha=0.3)
    delta_miou = args.miou_stego - args.miou_clean
    ax.text(0.02, 0.97, f"Utility cost: {delta_miou:+.2f} pp",
            transform=ax.transAxes, ha="left", va="top",
            fontsize=10, bbox=dict(boxstyle="round,pad=0.4",
                                   facecolor="white", edgecolor="gray", alpha=0.9))

    fig.suptitle("Inter-task steganography: secret recovery vs cover utility",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


def fig_drift(param_drift: float, out_path: Path):
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    ax.axis("off")
    ax.text(0.5, 0.72, "Relative L1 parameter drift", ha="center",
            va="center", fontsize=13, transform=ax.transAxes)
    ax.text(0.5, 0.45, f"{param_drift:.2e}", ha="center", va="center",
            fontsize=30, fontweight="bold", color="#d62728",
            transform=ax.transAxes)
    ax.text(0.5, 0.18,
            "Recovered encoder + secret_head vs original\n"
            "(LSB-induced perturbation propagates through layers)",
            ha="center", va="center", fontsize=10, color="#444",
            transform=ax.transAxes)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


def fig_psnr_comparison(args, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = ["Noisy input\n(no denoiser)", "Secret\ndenoiser",
              "Recovered\ndenoiser"]
    values = [args.psnr_floor, args.psnr_secret, args.psnr_recovered]
    bars = ax.bar(labels, values, color=["#7f7f7f", "#1f77b4", "#17becf"],
                  edgecolor="black", linewidth=0.6)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.3, f"{v:.2f} dB",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.axhline(args.psnr_floor, color="#7f7f7f", linestyle=":",
               linewidth=1, alpha=0.7)
    ax.set_ylabel("PSNR (dB)"); ax.set_ylim(0, max(values) + 5)
    ax.set_title(f"Denoising performance at $\\sigma$={args.noise_level:.0f}/255")
    ax.grid(axis="y", alpha=0.3)
    secret_gain = args.psnr_secret - args.psnr_floor
    rec_gain = args.psnr_recovered - args.psnr_floor
    ax.text(0.98, 0.97,
            f"Secret gain: +{secret_gain:.2f} dB over noise floor\n"
            f"Recovered gain: +{rec_gain:.2f} dB over noise floor",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=10, bbox=dict(boxstyle="round,pad=0.4",
                                   facecolor="white", edgecolor="gray", alpha=0.9))
    fig.tight_layout()
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt-dir", type=str, required=True)
    p.add_argument("--out-dir",  type=str, default="./figures_inter")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--epochs-secret", type=int, default=30)
    p.add_argument("--epochs-clean",  type=int, default=12)
    p.add_argument("--epochs-stego",  type=int, default=12)
    p.add_argument("--noise-level",   type=float, default=50.0)
    p.add_argument("--gfi-fraction",  type=float, default=0.30)
    p.add_argument("--encoder-width", type=int,   default=64)
    p.add_argument("--encoder-convs", type=int,   default=8)
    p.add_argument("--alpha", type=float, default=20.0)
    p.add_argument("--beta",  type=float, default=1.0)
    p.add_argument("--psnr-floor",     type=float, required=True)
    p.add_argument("--psnr-secret",    type=float, required=True)
    p.add_argument("--psnr-recovered", type=float, required=True)
    p.add_argument("--miou-stego",     type=float, required=True)
    p.add_argument("--miou-clean",     type=float, required=True)
    p.add_argument("--param-drift",    type=float, required=True)
    args, unknown = p.parse_known_args()
    if unknown:
        print(f"Warning: ignoring unexpected arguments: {unknown}")

    ckpt_dir = Path(args.ckpt_dir)
    out_dir  = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    secret_ckpt = ckpt_dir / (
        f"secret_dncnn_e{args.epochs_secret}_n{int(args.noise_level)}"
        f"_w{args.encoder_width}_c{args.encoder_convs}_s{args.seed}.pt")
    clean_ckpt = ckpt_dir / (
        f"clean_seg_e{args.epochs_clean}_gfi{args.gfi_fraction}"
        f"_w{args.encoder_width}_c{args.encoder_convs}_s{args.seed}.pt")
    stego_ckpt = ckpt_dir / (
        f"stego_inter_e{args.epochs_stego}_a{args.alpha}_b{args.beta}"
        f"_gfi{args.gfi_fraction}_w{args.encoder_width}"
        f"_c{args.encoder_convs}_s{args.seed}.pt")

    for path in (secret_ckpt, clean_ckpt, stego_ckpt):
        if not path.exists():
            raise FileNotFoundError(f"Missing: {path}")

    secret_losses = load_losses(secret_ckpt)
    clean_losses  = load_losses(clean_ckpt)
    stego_blob = torch.load(stego_ckpt, map_location="cpu", weights_only=False)
    stego_losses = stego_blob.get("losses", stego_blob.get("logs", []))
    if isinstance(stego_losses, dict):
        stego_losses = stego_losses.get("loss_total", [])

    if not secret_losses or not clean_losses or not stego_losses:
        print(f"  secret_losses: {len(secret_losses)} epochs")
        print(f"  clean_losses:  {len(clean_losses)} epochs")
        print(f"  stego_losses:  {len(stego_losses)} epochs")

    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    })

    fig_loss_curves(secret_losses, clean_losses, stego_losses,
                    out_dir / "01_loss_curves.png")
    fig_summary(args, out_dir / "02_summary.png")
    fig_drift(args.param_drift, out_dir / "03_drift.png")
    fig_psnr_comparison(args, out_dir / "04_psnr_comparison.png")

    print(f"\nAll figures written to: {out_dir}")


if __name__ == "__main__":
    main()

Writing /content/CS682-Project/make_plots_inter.py


In [11]:
!python /content/CS682-Project/make_plots_inter.py \
    --ckpt-dir /content/CS682-Project/checkpoints_inter \
    --out-dir  /content/CS682-Project/figures_inter \
    --epochs-secret 30 --epochs-clean 12 --epochs-stego 12 \
    --noise-level 50 --gfi-fraction 0.30 \
    --encoder-width 64 --encoder-convs 8 --seed 0 \
    --alpha 20.0 --beta 1.0 \
    --psnr-floor 15.01 \
    --psnr-secret 25.33 \
    --psnr-recovered 21.70 \
    --miou-stego 43.56 \
    --miou-clean 52.77 \
    --param-drift 3.071e-03

Saved: /content/CS682-Project/figures_inter/01_loss_curves.png
Saved: /content/CS682-Project/figures_inter/02_summary.png
Saved: /content/CS682-Project/figures_inter/03_drift.png
Saved: /content/CS682-Project/figures_inter/04_psnr_comparison.png

All figures written to: /content/CS682-Project/figures_inter


##if restarts

In [ ]:
# === Optional: reset caches before a fresh run ===
# Uncomment whichever level applies, then re-comment after the run.

# # Level 1: bytecode cache (always safe; near-zero cost)
# !find /content/CS682-Project -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null

# # Level 3a: stego + clean checkpoints (use after editing core/*.py or model)
# !rm -f /content/CS682-Project/checkpoints_inter/stego_inter_*.pt
# !rm -f /content/CS682-Project/checkpoints_inter/clean_seg_*.pt

# # Level 3b: also wipe secret + figures + metrics (full reset)
# !rm -f /content/CS682-Project/checkpoints_inter/secret_dncnn_*.pt
# !rm -rf /content/CS682-Project/figures_inter
# !rm -f /content/CS682-Project/sweep_metrics.jsonl

##fix relu

In [ ]:
file_path = "models/dncnn_dual.py"

with open(file_path, "r") as f:
    content = f.read()

# Replace nn.ReLU(inplace=True) with nn.ReLU()
new_content = content.replace("nn.ReLU(inplace=True)", "nn.ReLU()")

with open(file_path, "w") as f:
    f.write(new_content)

print(f"Successfully modified {file_path} to remove inplace=True from ReLU activations.")

Successfully modified models/dncnn_dual.py to remove inplace=True from ReLU activations.


##new gfi_seq.py modified

In [ ]:
"""Gradient-based filter insertion on arbitrary nn.Sequential stacks (Conv2d only indexed).

Patched: insert_one_filter_seq and remove_filter_seq now also resize the
BatchNorm2d that immediately follows the modified conv (assumes the standard
Conv -> BN -> activation pattern used in DnCNN/VGG-style encoders).
"""

from __future__ import annotations

import copy
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn

from core.gfi import InsertionSpec


def conv_indices_in_seq(seq: nn.Sequential) -> List[int]:
    return [i for i, m in enumerate(seq) if isinstance(m, nn.Conv2d)]


def _next_conv_index(seq: nn.Sequential, after_idx: int) -> Optional[int]:
    for i in range(after_idx + 1, len(seq)):
        if isinstance(seq[i], nn.Conv2d):
            return i
    return None


def _resize_bn_at(
    seq: nn.Sequential,
    idx: int,
    new_num_features: int,
    position: int,
    mode: str,
) -> nn.Sequential:
    """If seq[idx] is a BatchNorm2d, replace it with one sized to new_num_features.

    mode='insert': old size = new_num_features - 1; the channel at `position`
        is initialized to BN identity (weight=1, bias=0, mean=0, var=1).
    mode='remove': old size = new_num_features + 1; the channel at `position`
        is dropped.

    If seq[idx] is not a BatchNorm2d (e.g. activation, or end-of-sequence),
    this is a no-op.
    """
    if idx >= len(seq) or not isinstance(seq[idx], nn.BatchNorm2d):
        return seq
    old = seq[idx]
    device = old.weight.device if old.weight is not None else torch.device("cpu")
    dtype = old.weight.dtype if old.weight is not None else torch.float32
    new = nn.BatchNorm2d(
        new_num_features,
        eps=old.eps, momentum=old.momentum,
        affine=old.affine, track_running_stats=old.track_running_stats,
    ).to(device=device, dtype=dtype)

    if mode == "insert":
        for src_attr in ("weight", "bias", "running_mean", "running_var"):
            src = getattr(old, src_attr)
            dst = getattr(new, src_attr)
            if src is None or dst is None:
                continue
            if position > 0:
                dst.data[:position].copy_(src.data[:position])
            init_val = 1.0 if src_attr in ("weight", "running_var") else 0.0
            dst.data[position].fill_(init_val)
            if position < src.numel():
                dst.data[position + 1:].copy_(src.data[position:])
    elif mode == "remove":
        for src_attr in ("weight", "bias", "running_mean", "running_var"):
            src = getattr(old, src_attr)
            dst = getattr(new, src_attr)
            if src is None or dst is None:
                continue
            if position > 0:
                dst.data[:position].copy_(src.data[:position])
            if position + 1 < src.numel():
                dst.data[position:].copy_(src.data[position + 1:])
    else:
        raise ValueError(f"unknown mode: {mode}")

    if old.num_batches_tracked is not None and new.num_batches_tracked is not None:
        new.num_batches_tracked.copy_(old.num_batches_tracked)

    mods = list(seq.children())
    mods[idx] = new
    return nn.Sequential(*mods)


def insert_one_filter_seq(
    seq: nn.Sequential,
    conv_rank: int,
    position: int,
    rng: torch.Generator,
) -> nn.Sequential:
    """Insert one interference filter at conv_rank / position (0..out_ch).

    Updates: this conv (out += 1), the BN immediately after (num_features += 1
    if present), and the next conv if any (in += 1).
    """
    conv_ix_list = conv_indices_in_seq(seq)
    fi = conv_ix_list[conv_rank]
    conv = seq[fi]
    assert isinstance(conv, nn.Conv2d)
    device = conv.weight.device
    dtype = conv.weight.dtype
    in_ch, out_ch, kh, kw = conv.in_channels, conv.out_channels, conv.kernel_size[0], conv.kernel_size[1]

    new_conv = nn.Conv2d(
        in_ch, out_ch + 1, kernel_size=kh, padding=conv.padding[0], bias=conv.bias is not None
    ).to(device=device, dtype=dtype)
    w_old = conv.weight.data
    b_old = conv.bias.data if conv.bias is not None else None
    w_new = new_conv.weight.data
    b_new = new_conv.bias.data if new_conv.bias is not None else None

    if position > 0:
        w_new[:position].copy_(w_old[:position])
    if position < out_ch:
        w_new[position + 1 :].copy_(w_old[position:])
    nn.init.normal_(w_new[position : position + 1], 0, 0.02)
    if b_new is not None:
        if position > 0:
            b_new[:position].copy_(b_old[:position])
        if position < out_ch:
            b_new[position + 1 :].copy_(b_old[position:])
        b_new[position].zero_()

    mods = list(seq.children())
    mods[fi] = new_conv
    new_seq = nn.Sequential(*mods)

    # Resize BN immediately after this conv (Conv -> BN -> ReLU pattern)
    new_seq = _resize_bn_at(new_seq, fi + 1, out_ch + 1, position, mode="insert")

    next_fi = _next_conv_index(new_seq, fi)
    if next_fi is not None:
        nconv = new_seq[next_fi]
        assert isinstance(nconv, nn.Conv2d)
        new_next = nn.Conv2d(
            in_channels=out_ch + 1,
            out_channels=nconv.out_channels,
            kernel_size=nconv.kernel_size[0],
            padding=nconv.padding[0],
            bias=nconv.bias is not None,
        ).to(device=device, dtype=dtype)
        wn_old = nconv.weight.data
        wn_new = new_next.weight.data
        if position > 0:
            wn_new[:, :position].copy_(wn_old[:, :position])
        nn.init.normal_(wn_new[:, position : position + 1], 0, 0.02)
        if position < out_ch:
            wn_new[:, position + 1 :].copy_(wn_old[:, position:])
        if new_next.bias is not None and nconv.bias is not None:
            new_next.bias.data.copy_(nconv.bias.data)
        mods2 = list(new_seq.children())
        mods2[next_fi] = new_next
        new_seq = nn.Sequential(*mods2)

    return new_seq


def remove_filter_seq(seq: nn.Sequential, conv_rank: int, position: int) -> nn.Sequential:
    conv_ix_list = conv_indices_in_seq(seq)
    fi = conv_ix_list[conv_rank]
    conv = seq[fi]
    assert isinstance(conv, nn.Conv2d)
    device = conv.weight.device
    dtype = conv.weight.dtype
    in_ch, out_ch, kh, kw = conv.in_channels, conv.out_channels, conv.kernel_size[0], conv.kernel_size[1]
    assert 0 <= position < out_ch

    new_conv = nn.Conv2d(
        in_ch, out_ch - 1, kernel_size=kh, padding=conv.padding[0], bias=conv.bias is not None
    ).to(device=device, dtype=dtype)
    w_old = conv.weight.data
    b_old = conv.bias.data if conv.bias is not None else None
    w_new = new_conv.weight.data
    b_new = new_conv.bias.data if new_conv.bias is not None else None
    if position > 0:
        w_new[:position].copy_(w_old[:position])
    if position + 1 < out_ch:
        w_new[position:].copy_(w_old[position + 1 :])
    if b_new is not None and b_old is not None:
        if position > 0:
            b_new[:position].copy_(b_old[:position])
        if position + 1 < out_ch:
            b_new[position:].copy_(b_old[position + 1 :])

    mods = list(seq.children())
    mods[fi] = new_conv
    new_seq = nn.Sequential(*mods)

    # Resize BN immediately after this conv (Conv -> BN -> ReLU pattern)
    new_seq = _resize_bn_at(new_seq, fi + 1, out_ch - 1, position, mode="remove")

    next_fi = _next_conv_index(new_seq, fi)
    if next_fi is not None:
        nconv = new_seq[next_fi]
        assert isinstance(nconv, nn.Conv2d)
        new_next = nn.Conv2d(
            in_channels=out_ch - 1,
            out_channels=nconv.out_channels,
            kernel_size=nconv.kernel_size[0],
            padding=nconv.padding[0],
            bias=nconv.bias is not None,
        ).to(device=device, dtype=dtype)
        wn_old = nconv.weight.data
        wn_new = new_next.weight.data
        if position > 0:
            wn_new[:, :position].copy_(wn_old[:, :position])
        if position + 1 < out_ch:
            wn_new[:, position:].copy_(wn_old[:, position + 1 :])
        if new_next.bias is not None and nconv.bias is not None:
            new_next.bias.data.copy_(nconv.bias.data)
        mods2 = list(new_seq.children())
        mods2[next_fi] = new_next
        new_seq = nn.Sequential(*mods2)

    return new_seq


def _final_interference_indices(original_positions_desc: List[int]) -> List[int]:
    final_set = set()
    for p in original_positions_desc:
        final_set = {i + 1 if i >= p else i for i in final_set}
        final_set.add(p)
    return sorted(final_set)


def apply_insertions_seq(
    encoder: nn.Sequential,
    specs: Sequence[InsertionSpec],
    rng: torch.Generator,
) -> Tuple[nn.Sequential, Dict[int, torch.Tensor]]:
    by_rank: Dict[int, List[int]] = {}
    for s in specs:
        by_rank.setdefault(s.conv_rank, []).append(s.position)
    ordered: List[InsertionSpec] = []
    for rank in sorted(by_rank.keys()):
        for pos in sorted(by_rank[rank], reverse=True):
            ordered.append(InsertionSpec(conv_rank=rank, position=pos))

    rank_to_final_indices: Dict[int, List[int]] = {}
    for rank in by_rank:
        rank_to_final_indices[rank] = _final_interference_indices(sorted(by_rank[rank], reverse=True))

    seq = encoder
    for s in ordered:
        seq = insert_one_filter_seq(seq, s.conv_rank, s.position, rng)

    conv_masks: Dict[int, torch.Tensor] = {}
    conv_ix_list = conv_indices_in_seq(seq)
    for rank, fi in enumerate(conv_ix_list):
        conv = seq[fi]
        assert isinstance(conv, nn.Conv2d)
        d = conv.out_channels
        bits = torch.ones(d, dtype=torch.uint8)
        for idx in rank_to_final_indices.get(rank, []):
            if 0 <= idx < d:
                bits[idx] = 0
        conv_masks[rank] = bits
    return seq, conv_masks


def count_total_insertion_positions_seq(encoder: nn.Sequential) -> int:
    total = 0
    for fi in conv_indices_in_seq(encoder):
        conv = encoder[fi]
        assert isinstance(conv, nn.Conv2d)
        total += conv.out_channels + 1
    return total


def compute_position_importance_seq(
    model: nn.Module,
    encoder_attr: str,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    loss_forward: Callable[[nn.Module, torch.Tensor, torch.Tensor], torch.Tensor],
    max_batches: Optional[int] = None,
) -> Dict[Tuple[int, int], float]:
    """Eq. (2)-(4) using arbitrary encoder sequential inside model."""
    m = copy.deepcopy(model).to(device)
    m.train()
    seq = getattr(m, encoder_attr)
    assert isinstance(seq, nn.Sequential)
    conv_ix_list = conv_indices_in_seq(seq)

    grad_sum: Dict[int, Optional[torch.Tensor]] = {fi: None for fi in conv_ix_list}

    n_batches = 0
    for xb, yb in data_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        m.zero_grad(set_to_none=True)
        loss = loss_forward(m, xb, yb)
        loss.backward()
        for fi in conv_ix_list:
            conv = seq[fi]
            assert isinstance(conv, nn.Conv2d)
            g = conv.weight.grad
            if g is None:
                continue
            acc = torch.abs(g.detach())
            prev = grad_sum[fi]
            grad_sum[fi] = acc.clone() if prev is None else prev + acc
        n_batches += 1
        if max_batches is not None and n_batches >= max_batches:
            break

    importance: Dict[Tuple[int, int], float] = {}
    for rank, fi in enumerate(conv_ix_list):
        acc = grad_sum[fi]
        seq_cur = getattr(m, encoder_attr)
        conv = seq_cur[fi]
        assert isinstance(conv, nn.Conv2d)
        if acc is None:
            acc = torch.zeros_like(conv.weight)
        d = acc.shape[0]
        w = acc.view(d, -1).mean(dim=1)
        for j in range(d + 1):
            if j == 0:
                p = float(w[0].item())
            elif j == d:
                p = float(w[d - 1].item())
            else:
                p = 0.5 * (float(w[j - 1].item()) + float(w[j].item()))
            importance[(rank, j)] = p
    return importance


def top_n_positions(importance: Dict[Tuple[int, int], float], n_select: int) -> List[InsertionSpec]:
    items = sorted(importance.items(), key=lambda kv: kv[1], reverse=True)
    specs: List[InsertionSpec] = []
    for (rank, j), _ in items[:n_select]:
        specs.append(InsertionSpec(conv_rank=rank, position=j))
    return specs